
# E. coli Genome: Download & Analysis (Interactive Notebook)

This notebook downloads the *Escherichia coli* K-12 MG1655 reference genome and performs basic exploratory analyses:

- Download the reference FASTA (RefSeq ASM584v2) from NCBI
- Parse sequences (chromosome + plasmids if present)
- Summary statistics (lengths, N50, GC%)
- **Adaptive plotting:** if only one sequence is present, we skip contig-length histograms and use GC% window histograms
- Base composition & GC skew
- Windowed GC% profile
- k-mer frequencies (configurable *k*)
- Simple ORF scan (naïve start/stop scan in all 6 frames)
- Histograms and line plots

> **Note:** Internet access is required the first time you run the download cell. Thereafter, the cached files are used.


In [ ]:

# If running on a clean environment, uncomment to install Biopython
# %pip install biopython --quiet

import os
import gzip
import io
import math
import urllib.request
from pathlib import Path
from collections import Counter, defaultdict

from Bio import SeqIO
from Bio.Seq import Seq

import numpy as np
import matplotlib.pyplot as plt

# Matplotlib defaults (publication-friendly but simple)
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = True

DATA_DIR = Path("data_ecoli")
DATA_DIR.mkdir(exist_ok=True, parents=True)

# RefSeq Escherichia coli K-12 substr. MG1655 (ASM584v2) FASTA (gzipped)
NCBI_FASTA_URL = ("https://ftp.ncbi.nlm.nih.gov/genomes/refseq/bacteria/Escherichia_coli/"
                  "reference/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.fna.gz")
LOCAL_FASTA_GZ = DATA_DIR / "GCF_000005845.2_ASM584v2_genomic.fna.gz"
LOCAL_FASTA = DATA_DIR / "GCF_000005845.2_ASM584v2_genomic.fna"

print("Data directory:", DATA_DIR.resolve())



## 1) Download the E. coli reference genome

This cell downloads the RefSeq FASTA file (if not already cached) from NCBI FTP.


In [ ]:

def download_if_needed(url: str, dst: Path):
    if dst.exists():
        print(f"✅ Already present: {dst.name}")
        return
    print(f"⬇️ Downloading {url} ...")
    urllib.request.urlretrieve(url, dst)
    print(f"✅ Saved to {dst}")

download_if_needed(NCBI_FASTA_URL, LOCAL_FASTA_GZ)

# Decompress if needed
if not LOCAL_FASTA.exists():
    print("🔧 Decompressing FASTA ...")
    import gzip
    with gzip.open(LOCAL_FASTA_GZ, "rb") as fin, open(LOCAL_FASTA, "wb") as fout:
        fout.write(fin.read())
    print(f"✅ Decompressed: {LOCAL_FASTA.name}")
else:
    print(f"✅ Already decompressed: {LOCAL_FASTA.name}")



## 2) Load sequences

We parse the FASTA to get one or more sequences (chromosome, plasmids).

In [ ]:

records = list(SeqIO.parse(str(LOCAL_FASTA), "fasta"))
assert len(records) > 0, "No sequences loaded. Check your internet connection and try re-running the download cell."

print(f"Loaded {len(records)} sequence(s).")
for i, rec in enumerate(records, 1):
    desc = rec.description[:80] + ("..." if len(rec.description) > 80 else "")
    print(f"{i:>2}. {rec.id} | length={len(rec.seq):,} | {desc}")



## 3) Basic statistics

We compute sequence lengths, total length, GC content, and N50. 

In [ ]:

def gc_content(seq: str) -> float:
    s = seq.upper()
    gc = s.count("G") + s.count("C")
    atgc = s.count("A") + s.count("T") + s.count("G") + s.count("C")
    return (gc / atgc * 100.0) if atgc else float("nan")

lengths = np.array([len(r.seq) for r in records], dtype=np.int64)
total_length = int(lengths.sum())

gc_all = gc_content("".join(str(r.seq) for r in records))

def n50(lengths):
    arr = np.sort(lengths)[::-1]
    csum = np.cumsum(arr)
    half = csum[-1] / 2
    return int(arr[np.where(csum >= half)[0][0]])

stats = {
    "num_sequences": len(records),
    "total_length": total_length,
    "min_length": int(lengths.min()),
    "max_length": int(lengths.max()),
    "mean_length": float(lengths.mean()),
    "N50": n50(lengths),
    "GC_percent_total": gc_all,
}

stats



### Adaptive histogram

- If multiple sequences/contigs are present, we show the contig length histogram.
- If only one sequence is present (typical for *E. coli* K-12), we show a histogram of windowed GC% values instead.


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

genome = "".join(str(r.seq).upper() for r in records)

def sliding_gc_content(seq: str, window: int = 5000, step: int = 1000):
    gc_vals = []
    centers = []
    for i in range(0, len(seq) - window + 1, step):
        w = seq[i:i+window]
        s = w.upper()
        gc = (s.count("G") + s.count("C"))
        atgc = (s.count("A") + s.count("T") + s.count("G") + s.count("C"))
        gc_percent = (gc/atgc*100.0) if atgc else float("nan")
        gc_vals.append(gc_percent)
        centers.append(i + window//2)
    return np.array(centers), np.array(gc_vals)

if len(records) > 1:
    plt.figure()
    plt.hist(lengths, bins=min(len(lengths), 30))
    plt.xlabel("Sequence length (bp)")
    plt.ylabel("Count")
    plt.title("Histogram of contig/sequence lengths")
    plt.show()
else:
    centers, gc_vals = sliding_gc_content(genome, window=5000, step=1000)
    plt.figure()
    plt.hist(gc_vals[~np.isnan(gc_vals)], bins=30)
    plt.xlabel("GC% per window")
    plt.ylabel("Count")
    plt.title("Histogram of windowed GC% (window=5kb, step=1kb)")
    plt.show()



## 4) Base composition and GC skew

We compute per-base composition on the concatenated genome and show a simple GC skew profile:

- GC skew = (G - C) / (G + C) in a sliding window


In [ ]:

from collections import Counter
counts = Counter(genome)
counts_dict = {b: counts.get(b, 0) for b in ["A","C","G","T","N"]}
counts_dict


In [ ]:

def sliding_gc_skew(seq: str, window: int = 2000, step: int = 500):
    skew_vals = []
    pos = []
    for i in range(0, len(seq) - window + 1, step):
        w = seq[i:i+window]
        g = w.count("G")
        c = w.count("C")
        denom = g + c
        skew = (g - c) / denom if denom else 0.0
        skew_vals.append(skew)
        pos.append(i + window//2)
    return np.array(pos), np.array(skew_vals)

pos, skew = sliding_gc_skew(genome, window=2000, step=500)

plt.figure()
plt.plot(pos, skew)
plt.xlabel("Position (bp)")
plt.ylabel("GC skew")
plt.title("GC Skew (window=2000, step=500)")
plt.show()



## 5) Windowed GC% profile (line)


In [ ]:

centers, gc_vals = (lambda seq: (
    np.array([i + 2500 for i in range(0, len(seq) - 5000 + 1, 1000)]),
    np.array([
        (w.count("G")+w.count("C")) / max(1, (w.count("A")+w.count("T")+w.count("G")+w.count("C"))) * 100.0
        for w in (seq[i:i+5000] for i in range(0, len(seq) - 5000 + 1, 1000))
    ])
))(genome)

plt.figure()
plt.plot(centers, gc_vals)
plt.xlabel("Position (bp)")
plt.ylabel("GC%")
plt.title("Windowed GC% (window=5000, step=1000)")
plt.show()



## 6) k-mer frequencies

Count canonical k-mers (treating a k-mer and its reverse complement as the same). Adjust `k` as needed.


In [ ]:

from Bio.Seq import Seq

def revcomp(s: str) -> str:
    return str(Seq(s).reverse_complement())

def canonical_kmer(s: str) -> str:
    rc = revcomp(s)
    return min(s, rc)

def kmer_counts(seq: str, k: int = 4):
    seq = seq.upper()
    valid = set("ACGT")
    cnt = Counter()
    for i in range(len(seq) - k + 1):
        kmer = seq[i:i+k]
        if set(kmer) <= valid:
            cnt[canonical_kmer(kmer)] += 1
    return cnt

k = 4  # change k here if desired
kcnt = kmer_counts(genome, k=k)
# Show top 20
top20 = sorted(kcnt.items(), key=lambda x: x[1], reverse=True)[:20]
top20


In [ ]:

labels, values = zip(*sorted(kcnt.items(), key=lambda x: x[1], reverse=True)[:20])
x = np.arange(len(labels))

plt.figure(figsize=(10,4))
plt.bar(x, values)
plt.xticks(x, labels, rotation=90)
plt.ylabel("Count")
plt.title(f"Top {len(labels)} canonical {k}-mers")
plt.tight_layout()
plt.show()



## 7) Simple ORF scan (6 frames)

This is a **naïve** ORF finder scanning all six frames using typical bacterial start/stop codons:
- Starts: ATG, GTG, TTG
- Stops: TAA, TAG, TGA

Use this for exploratory analysis only (not for annotation-grade calls).


In [ ]:

STARTS = {"ATG","GTG","TTG"}
STOPS = {"TAA","TAG","TGA"}

def find_orfs(seq: str, min_aa_len: int = 120):
    seq = seq.upper()
    from Bio.Seq import Seq as _Seq
    def _revcomp(s): return str(_Seq(s).reverse_complement())
    rc = _revcomp(seq)
    results = []

    def scan_one(s: str, strand: int):
        n = len(s)
        i = 0
        while i <= n - 3:
            codon = s[i:i+3]
            if codon in STARTS:
                j = i + 3
                while j <= n - 3:
                    stop = s[j:j+3]
                    if stop in STOPS:
                        aa_len = (j - i) // 3
                        if aa_len >= min_aa_len:
                            if strand == +1:
                                start, end = i, j+3
                            else:
                                start_rc = n - (j+3)
                                end_rc = n - i
                                start, end = start_rc, end_rc
                            results.append({
                                "strand": strand,
                                "start": start,
                                "end": end,
                                "length_aa": aa_len
                            })
                        i = j + 3
                        break
                    j += 3
            i += 3

    scan_one(seq, +1)
    scan_one(rc, -1)
    return sorted(results, key=lambda d: (d["strand"], d["start"]))

orfs = find_orfs(genome, min_aa_len=120)
len(orfs), orfs[:3]


In [ ]:

# ORF length distribution (AA)
orf_lengths = np.array([o["length_aa"] for o in orfs])
plt.figure()
plt.hist(orf_lengths, bins=50)
plt.xlabel("ORF length (aa)")
plt.ylabel("Count")
plt.title("Naïve ORF length distribution")
plt.show()

# Quick summary
{
    "num_orfs_minlen120aa": int(len(orfs)),
    "median_orf_len_aa": float(np.median(orf_lengths)) if len(orf_lengths) else None,
    "max_orf_len_aa": int(orf_lengths.max()) if len(orf_lengths) else None,
}



### Save ORF table (optional)


In [ ]:

import csv

ORF_CSV = DATA_DIR / "naive_orfs_min120aa.csv"
with open(ORF_CSV, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["strand","start","end","length_aa"])
    for o in orfs:
        w.writerow([o["strand"], o["start"], o["end"], o["length_aa"]])
print(f"Saved: {ORF_CSV.resolve()}")



## 8) Next steps

- Compare GC/GC-skew features to known replication origin/terminus
- Overlay annotated genes from GenBank (`.gbff`) to validate ORF calls
- Compute codon usage bias and CAI per ORF
- Add prophage or island prediction with external tools
- Run alignment-based analyses against other *E. coli* strains
